# A2 — Reconhecimento Semântico em Publicidade Visual com CLIP

**Projeto:** EcoVision AI  
**Dataset:** ADS-16 (Computational Advertising Dataset)  
**Modelo:** `openai/clip-vit-base-patch32`

## Objetivos

Esta atividade utiliza o CLIP **sem treinamento adicional** para:

1. extrair embeddings multimodais de imagens e textos;
2. calcular similaridade semântica por cosine similarity;
3. ranquear pelo menos 20 conceitos por frequência semântica;
4. justificar um threshold de similaridade;
5. visualizar os 5 conceitos mais frequentes com exemplos;
6. executar pelo menos 8 buscas semânticas por texto;
7. analisar criticamente as recuperações;
8. explicar por que o pré-treinamento contrastivo permite zero-shot/retrieval;
9. comparar a consulta textual do CLIP com a tokenização de sequências no BERT.

## Requisitos de execução

- **Ambiente recomendado:** Google Colab com GPU T4
- **GPU:** T4 (16 GB de VRAM)
- **RAM recomendada:** 8 GB ou mais
- **Espaço livre recomendado:** pelo menos 3 GB para dataset, cache e artefatos
- **Batch size CLIP:** 16
- **Corpus utilizado:** 600 imagens representativas
- **Consultas semânticas:** 8
- **Descrições/conceitos:** 24
- **Tempo estimado em T4:** aproximadamente 5–15 minutos, dependendo de download/cache
- **Treinamento de modelo:** nenhum; somente inferência com pesos pré-treinados

> O notebook reutiliza os módulos organizados em `src/clip/`.  
> Em uma execução nova, o corpus é preparado automaticamente e os artefatos CLIP são recalculados quando necessário.

## 1. Preparação do ambiente

O CLIP possui dois encoders principais:

```text
Imagem ──→ Vision Encoder ──→ embedding visual
                                  │
                                  │ espaço multimodal compartilhado
                                  │
Texto  ──→ Text Encoder   ──→ embedding textual
```

O objetivo desta atividade não é treinar uma nova rede. Utilizamos o conhecimento que o CLIP adquiriu durante o pré-treinamento para comparar imagens e linguagem natural diretamente.

In [ ]:
import sys
import subprocess

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "transformers",
        "kagglehub",
        "pandas",
        "matplotlib",
        "pillow",
        "scikit-learn",
    ],
    check=True,
)

print("Dependências verificadas.")

In [ ]:
from pathlib import Path
import os
import sys

def encontrar_raiz_projeto():
    candidatos = []

    cwd = Path.cwd().resolve()
    candidatos.extend([cwd, *cwd.parents])

    candidatos.extend([
        Path("/content/EcoVision-AI-organized"),
        Path("/content/EcoVision-AI"),
    ])

    for candidato in candidatos:
        if (
            (candidato / "src" / "clip").exists()
            and (candidato / "src" / "common").exists()
        ):
            return candidato

    # No Colab, tenta localizar o projeto no Google Drive.
    try:
        from google.colab import drive

        drive.mount("/content/drive")

        mydrive = Path("/content/drive/MyDrive")

        for nome in ["EcoVision-AI-organized", "EcoVision-AI"]:
            for candidato in mydrive.glob(f"**/{nome}"):
                if (
                    (candidato / "src" / "clip").exists()
                    and (candidato / "src" / "common").exists()
                ):
                    return candidato.resolve()

    except Exception:
        pass

    raise FileNotFoundError(
        "Não encontrei a raiz do EcoVision-AI. "
        "No Colab, coloque a pasta do projeto em /content "
        "ou no Google Drive."
    )

PROJECT_ROOT = encontrar_raiz_projeto()

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Raiz do projeto:", PROJECT_ROOT)
print("Diretório atual:", Path.cwd())

In [ ]:
import torch

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("PyTorch:", torch.__version__)
print("Dispositivo:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    total_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"VRAM disponível: {total_vram:.2f} GB")
else:
    print(
        "O pipeline também funciona em CPU, "
        "mas a extração dos embeddings será mais lenta."
    )

## 2. Preparação do corpus ADS-16

A atividade permite utilizar o corpus completo ou um subconjunto representativo com pelo menos 500 imagens.

No EcoVision AI foi construído um corpus com **600 imagens**:

```text
120 usuários
×
5 imagens por usuário
=
600 imagens
```

A seleção alterna aproximadamente:

```text
3 POS + 2 NEG
2 POS + 3 NEG
```

entre usuários, produzindo um corpus global praticamente equilibrado entre os grupos `POS` e `NEG`.

### Deduplicação

Antes da seleção foi aplicada deduplicação global usando **SHA-256**.

SHA-256 detecta arquivos com conteúdo binário exatamente igual:

```text
imagem
 ↓
SHA-256
 ↓
hash
 ↓
hash já existe?
 ├─ sim → duplicata exata
 └─ não → manter
```

No experimento executado, o corpus de origem considerado continha 2.396 imagens, das quais 47 eram duplicatas exatas, deixando 2.349 arquivos únicos antes da seleção final.

> Limitação: SHA-256 remove duplicatas **exatas**, mas não necessariamente imagens visualmente quase idênticas que foram redimensionadas, recomprimidas ou levemente modificadas. Isso será considerado na análise crítica.

In [ ]:
import pandas as pd
import subprocess
import sys

MANIFEST_PATH = PROJECT_ROOT / "data" / "ads16_clip_manifest.csv"
CORPUS_DIR = PROJECT_ROOT / "data" / "ads16_clip"

def manifest_valido():
    if not MANIFEST_PATH.exists():
        return False

    try:
        df = pd.read_csv(MANIFEST_PATH)
    except Exception:
        return False

    if len(df) < 500 or "arquivo" not in df.columns:
        return False

    # Detecta manifest copiado de outra pasta/máquina,
    # cujos caminhos absolutos não existem mais.
    amostra = df["arquivo"].head(min(10, len(df))).tolist()

    return bool(amostra) and all(Path(caminho).exists() for caminho in amostra)


corpus_recriado = False

if manifest_valido():
    print("Corpus ADS-16 já preparado e com caminhos válidos.")
else:
    print("Corpus ausente ou manifest com caminhos antigos.")
    print("Preparando novamente o subconjunto representativo...")

    subprocess.run(
        [sys.executable, "-m", "src.clip.prepare_corpus"],
        cwd=PROJECT_ROOT,
        check=True,
    )

    corpus_recriado = True

assert manifest_valido(), "Falha ao preparar o corpus ADS-16."

manifest = pd.read_csv(MANIFEST_PATH)

print()
print("Imagens selecionadas:", len(manifest))
print("Usuários:", manifest["usuario"].nunique())
print()
print("Distribuição POS/NEG:")
print(manifest["sentimento"].value_counts())
print()
print("Hashes únicos:", manifest["sha256"].nunique())

In [ ]:
# Conferência de duplicatas exatas no corpus final.
duplicatas_exatas = (
    len(manifest)
    - manifest["sha256"].nunique()
)

print("Duplicatas exatas no corpus final:", duplicatas_exatas)

assert len(manifest) >= 500
assert duplicatas_exatas == 0

print("Validação do corpus: OK")

## 3. O que é CLIP?

CLIP significa **Contrastive Language–Image Pre-training**.

Durante o pré-treinamento, o modelo aprende a aproximar representações de imagens e textos semanticamente relacionados.

Considere um batch com pares corretos:

```text
imagem 1 ↔ texto 1
imagem 2 ↔ texto 2
imagem 3 ↔ texto 3
...
```

O treinamento constrói embeddings e compara todos contra todos:

```text
               Texto 1   Texto 2   Texto 3
Imagem 1          ↑
Imagem 2                    ↑
Imagem 3                              ↑
```

Os pares corretos devem receber similaridade maior, enquanto combinações incorretas devem receber similaridade menor.

Esse treinamento é **contrastivo**: o modelo aprende distinguindo pares compatíveis de pares incompatíveis.

### Por que isso habilita zero-shot?

Depois do pré-treinamento:

```text
imagem → embedding ┐
                   ├→ mesmo espaço semântico
texto  → embedding ┘
```

Assim, uma nova descrição como:

```text
"a food product"
```

pode ser comparada diretamente às imagens, mesmo que não tenhamos treinado um classificador supervisionado específico para essa descrição.

Isso permite:

- classificação zero-shot;
- recuperação de imagens por texto;
- ranking semântico;
- busca multimodal.

## 4. Embeddings e cosine similarity

Cada imagem e texto é projetado para o mesmo espaço vetorial.

No modelo utilizado neste projeto, os embeddings multimodais possuem dimensão 512.

Depois aplicamos normalização L2:

\[
\hat{x} = \frac{x}{||x||_2}
\]

Com os dois vetores normalizados, o produto escalar equivale à cosine similarity:

\[
cos(A,B) =
\frac{A \cdot B}{||A||\,||B||}
\]

Como:

\[
||A|| = ||B|| = 1
\]

temos:

\[
cos(A,B) = A \cdot B
\]

No código isso permite calcular todas as similaridades com uma multiplicação matricial eficiente:

```python
image_embeddings @ text_embeddings.T
```

## 5. Ranking semântico com 24 conceitos

Foram utilizadas 24 descrições, ultrapassando o mínimo de 20 solicitado:

```text
a person
a human face
a man
a woman
a child
a car
a bicycle
a motorcycle
a food product
a beverage
clothing
shoes
a smartphone
a computer
an electronic device
furniture
an animal
a pet
outdoor scenery
a beach
a city or building
sports
text and logo
a packaged product
```

Para cada uma das 600 imagens, calculamos a similaridade com os 24 textos:

```text
600 imagens × 24 conceitos
=
14.400 comparações semânticas
```

In [ ]:
RESULTS_DIR = PROJECT_ROOT / "results" / "clip"

ARTEFATOS_RANKING = [
    RESULTS_DIR / "clip_image_embeddings.pt",
    RESULTS_DIR / "clip_concept_ranking.csv",
    RESULTS_DIR / "clip_concept_ranking.json",
    RESULTS_DIR / "clip_top5_concepts.png",
    RESULTS_DIR / "clip_top5_examples.png",
    RESULTS_DIR / "clip_config.json",
]

precisa_ranking = (
    corpus_recriado
    or not all(caminho.exists() for caminho in ARTEFATOS_RANKING)
)

if precisa_ranking:
    print("Executando pipeline principal do CLIP...")
    subprocess.run(
        [sys.executable, "-m", "src.clip.semantic_ranking"],
        cwd=PROJECT_ROOT,
        check=True,
    )
else:
    print("Artefatos do ranking já existem. Reutilizando resultados.")

In [ ]:
ranking = pd.read_csv(
    RESULTS_DIR / "clip_concept_ranking.csv"
)

ranking

### Resultado observado

Com threshold `0.22`, os cinco conceitos mais frequentes foram:

| Posição | Conceito | Frequência | % das 600 imagens |
|---|---|---:|---:|
| 1 | `an animal` | 133 | 22,17% |
| 2 | `a pet` | 109 | 18,17% |
| 3 | `a food product` | 101 | 16,83% |
| 4 | `a person` | 100 | 16,67% |
| 5 | `a human face` | 93 | 15,50% |

Um ponto importante é que esses números representam **correspondências semânticas acima do threshold**, não rótulos verdadeiros anotados manualmente. Uma mesma imagem também pode corresponder a vários conceitos.

In [ ]:
from IPython.display import display, Image as IPyImage

for nome in [
    "clip_top5_concepts.png",
    "clip_top5_examples.png",
]:
    caminho = RESULTS_DIR / nome

    if caminho.exists():
        print(nome)
        display(IPyImage(filename=str(caminho)))

## 6. Como o threshold foi escolhido?

Um score CLIP **não é uma probabilidade**.

Por exemplo:

```text
similaridade = 0.24
```

não significa:

```text
24% de probabilidade de a imagem conter o objeto
```

Ele é uma medida de proximidade no espaço de embeddings.

No experimento, a distribuição das 14.400 comparações apresentou aproximadamente:

| Estatística | Similaridade |
|---|---:|
| mínimo | 0,1149 |
| média | 0,1902 |
| mediana | 0,1890 |
| desvio padrão | 0,0218 |
| percentil 75 | 0,2031 |
| percentil 90 | 0,2177 |
| percentil 95 | 0,2284 |
| máximo | 0,2943 |

Por isso `0.22` foi escolhido como threshold operacional: ele fica próximo do percentil 90 da distribuição observada e mantém um número suficiente de correspondências para produzir um ranking útil.

A escolha é **empírica e específica deste corpus/modelo**, não um threshold universal do CLIP.

In [ ]:
# Recalcula a distribuição para que a justificativa possa
# ser reproduzida no ambiente atual.

import numpy as np
import torch

from src.clip.semantic_ranking import (
    carregar_clip,
    gerar_embeddings_texto,
)

image_embeddings = torch.load(
    RESULTS_DIR / "clip_image_embeddings.pt",
    map_location="cpu",
)

model, processor = carregar_clip()

text_embeddings = gerar_embeddings_texto(
    model,
    processor,
)

similarities = (
    image_embeddings
    @ text_embeddings.cpu().T
)

valores = similarities.flatten().numpy()

estatisticas = pd.DataFrame({
    "estatística": [
        "mínimo",
        "média",
        "mediana",
        "desvio padrão",
        "percentil 75",
        "percentil 90",
        "percentil 95",
        "máximo",
    ],
    "valor": [
        valores.min(),
        valores.mean(),
        np.median(valores),
        valores.std(),
        np.percentile(valores, 75),
        np.percentile(valores, 90),
        np.percentile(valores, 95),
        valores.max(),
    ],
})

estatisticas

## 7. Análise de sensibilidade do threshold

Também comparamos quatro valores:

```text
0.22
0.23
0.24
0.25
```

Quanto maior o threshold, mais seletivo fica o filtro.

Isso produz um trade-off:

```text
threshold menor
→ mais correspondências
→ maior cobertura
→ maior chance de falsos positivos

threshold maior
→ menos correspondências
→ maior seletividade
→ maior chance de perder correspondências úteis
```

In [ ]:
THRESHOLD_PATH = RESULTS_DIR / "clip_threshold_analysis.csv"

if (
    corpus_recriado
    or precisa_ranking
    or not THRESHOLD_PATH.exists()
):
    subprocess.run(
        [sys.executable, "-m", "src.clip.threshold_analysis"],
        cwd=PROJECT_ROOT,
        check=True,
    )

threshold_df = pd.read_csv(THRESHOLD_PATH)

resumo_threshold = (
    threshold_df
    .groupby("threshold", as_index=False)
    .agg(
        pares_acima_threshold=("frequencia", "sum")
    )
)

resumo_threshold["percentual_dos_14400_pares"] = (
    resumo_threshold["pares_acima_threshold"]
    / (600 * 24)
    * 100
)

resumo_threshold

No experimento original:

| Threshold | Pares acima do corte | % dos 14.400 pares |
|---:|---:|---:|
| 0,22 | 1.254 | 8,71% |
| 0,23 | 639 | 4,44% |
| 0,24 | 316 | 2,19% |
| 0,25 | 158 | 1,10% |

O salto entre `0.22` e `0.25` mostra que pequenas alterações no corte têm grande efeito sobre a quantidade de correspondências consideradas presentes.

Por isso o threshold deve ser documentado e justificado, e não escolhido como se fosse uma constante universal.

## 8. Busca semântica por linguagem natural

Na busca semântica não precisamos definir um threshold.

Para uma consulta:

```text
"a person outdoors"
```

fazemos:

```text
consulta textual
      ↓
CLIP Text Encoder
      ↓
embedding da consulta
      ↓
cosine similarity com as 600 imagens
      ↓
ordenação decrescente
      ↓
Top-5
```

As 8 consultas foram escolhidas para variar de conceitos concretos/genéricos até composições mais específicas e abstratas.

In [ ]:
SEARCH_JSON = RESULTS_DIR / "clip_semantic_search_results.json"
SEARCH_CSV = RESULTS_DIR / "clip_semantic_search_results.csv"

if (
    corpus_recriado
    or precisa_ranking
    or not SEARCH_JSON.exists()
    or not SEARCH_CSV.exists()
):
    subprocess.run(
        [sys.executable, "-m", "src.clip.semantic_search"],
        cwd=PROJECT_ROOT,
        check=True,
    )

search_df = pd.read_csv(SEARCH_CSV)

search_df[
    ["consulta", "posicao", "score", "usuario", "sentimento"]
]

In [ ]:
# Resumo dos scores Top-5 por consulta.
resumo_buscas = (
    search_df
    .groupby("consulta")
    .agg(
        melhor_score=("score", "max"),
        menor_score_top5=("score", "min"),
        score_medio_top5=("score", "mean"),
    )
    .reset_index()
)

resumo_buscas

In [ ]:
# Exibe as oito visualizações geradas pelo pipeline.
SEARCH_IMAGES_DIR = RESULTS_DIR / "semantic_search"

arquivos_busca = [
    "a_person.png",
    "an_animal.png",
    "a_food_product.png",
    "a_person_outdoors.png",
    "a_child_playing_sports.png",
    "a_person_using_an_electronic_device.png",
    "a_relaxing_outdoor_scene.png",
    "a_happy_social_moment.png",
]

for nome in arquivos_busca:
    caminho = SEARCH_IMAGES_DIR / nome

    if caminho.exists():
        print(nome)
        display(IPyImage(filename=str(caminho)))

## 9. Análise das oito consultas

A análise abaixo foi feita visualmente sobre os Top-5 produzidos pelo experimento.

| Consulta | Análise |
|---|---|
| `a person` | **Parcialmente boa.** Recupera imagens relacionadas a pessoas, mas também pode responder a representações humanas, elementos gráficos ou pistas contextuais em vez de uma pessoa claramente visível. |
| `an animal` | **Muito boa.** Os cinco primeiros resultados são fortemente coerentes com o conceito de animal. |
| `a food product` | **Boa, com ruído.** Os primeiros resultados representam bem alimentos/produtos alimentícios; pelo menos um resultado inferior do Top-5 é menos coerente. |
| `a person outdoors` | **Muito boa.** A composição “pessoa + ambiente externo” é recuperada de forma convincente em grande parte dos resultados. |
| `a child playing sports` | **Parcial.** O melhor resultado combina bem os elementos da consulta, mas outros recuperam apenas partes do conceito, como esporte/bola, sem satisfazer necessariamente “criança jogando”. |
| `a person using an electronic device` | **Fraca/parcial.** Alguns resultados contêm eletrônicos sem uma pessoa usando-os; outros contêm pessoas sem evidência clara da ação solicitada. |
| `a relaxing outdoor scene` | **Muito boa.** O CLIP recupera cenas naturais/externas visualmente compatíveis com o conceito abstrato de relaxamento. |
| `a happy social moment` | **Boa/parcial.** Há forte associação com felicidade e pessoas, embora nem todos os resultados expressem claramente uma interação social. |

### O que isso demonstra?

O CLIP não funciona como uma busca literal por palavras.

Ele tenta aproximar o **significado global** da consulta da representação visual.

Por isso uma consulta composta, como:

```text
"a child playing sports"
```

pode recuperar uma imagem que corresponda fortemente a `sports`, mas apenas fracamente a `child`.

Isso mostra uma limitação importante: similaridade semântica não garante que **todas as condições linguísticas da frase** estejam presentes simultaneamente.

## 10. Limitação observada — near-duplicates

A deduplicação SHA-256 eliminou arquivos binariamente idênticos.

Entretanto, durante a inspeção dos resultados foram observadas imagens visualmente muito semelhantes.

Isso ocorre porque:

```text
mesma imagem
+ resize/recompressão/pequena alteração
=
hash diferente
```

Portanto:

```text
SHA-256
→ bom para duplicata exata

Perceptual Hash / embeddings
→ mais apropriados para near-duplicates
```

Em uma versão futura, `pHash`, `dHash`, SSIM ou distância entre embeddings poderiam ser utilizados para reduzir redundância perceptual no corpus.

## 11. CLIP × BERT — consulta textual e tokenização

CLIP e BERT utilizam Transformers para texto, mas foram treinados para objetivos diferentes.

### BERT

Fluxo simplificado:

```text
texto
 ↓
tokenização
 ↓
input_ids
 ↓
attention_mask
 ↓
Transformer Encoder
 ↓
representações contextuais de linguagem
```

O BERT original foi pré-treinado principalmente com **Masked Language Modeling (MLM)** e também com **Next Sentence Prediction (NSP)**.

Ele aprende linguagem, mas não é treinado originalmente para colocar textos no mesmo espaço de embeddings que imagens.

### CLIP

No CLIP:

```text
texto
 ↓
tokenização
 ↓
input_ids + attention_mask
 ↓
Text Transformer
 ↓
text_projection
 ↓
embedding multimodal
```

A projeção final coloca o texto em um espaço compatível com o embedding gerado pelo encoder visual.

É justamente isso que possibilita:

```text
texto @ imagem
```

para busca semântica.

### Padding

Frases em um mesmo batch podem ter comprimentos diferentes.

Exemplo:

```text
"a car"
"a person using an electronic device"
```

Para montar um tensor retangular, as sequências são ajustadas para comprimentos compatíveis por **padding**.

Conceitualmente:

```text
frase curta: [token token PAD PAD PAD]
frase longa: [token token token token token]
```

### Attention Mask

A `attention_mask` informa quais posições representam tokens válidos e quais são apenas padding.

Exemplo:

```text
tokens:
[A, B, PAD, PAD]

attention_mask:
[1, 1, 0, 0]
```

Assim, o Transformer sabe que as posições de padding não devem contribuir como conteúdo real.

No nosso código CLIP isso aparece explicitamente:

```python
inputs = processor(
    text=CONCEPTS,
    return_tensors="pt",
    padding=True,
    truncation=True
)

input_ids = inputs["input_ids"]
attention_mask = inputs["attention_mask"]
```

Portanto, tanto BERT quanto o encoder textual do CLIP utilizam mecanismos de tokenização/mascaramento típicos de Transformers. A diferença fundamental aqui é **o objetivo de treinamento e o espaço final da representação**.

In [ ]:
# Demonstração prática da tokenização usada pelo CLIP.

consultas_demo = [
    "a car",
    "a person using an electronic device",
]

inputs_demo = processor(
    text=consultas_demo,
    return_tensors="pt",
    padding=True,
    truncation=True,
)

print("input_ids:")
print(inputs_demo["input_ids"])
print()
print("attention_mask:")
print(inputs_demo["attention_mask"])
print()
print("Shape input_ids:", tuple(inputs_demo["input_ids"].shape))
print("Shape attention_mask:", tuple(inputs_demo["attention_mask"].shape))

## 12. Zero-shot, ranking e semantic search: diferenças

Esses conceitos são relacionados, mas não idênticos.

### Zero-shot classification

Compara uma imagem a vários prompts/categorias e escolhe a descrição mais compatível:

```text
imagem
 ↓
["cat", "dog", "car", ...]
 ↓
maior similaridade
```

### Ranking por frequência semântica

Neste trabalho:

```text
600 imagens
×
24 conceitos
↓
scores
↓
threshold
↓
quantas imagens superaram o corte para cada conceito?
```

### Semantic search

Parte de uma consulta textual e busca as imagens mais próximas:

```text
texto
 ↓
embedding
 ↓
comparação com corpus
 ↓
Top-K imagens
```

O mesmo espaço multimodal do CLIP torna as três aplicações possíveis sem treinamento supervisionado específico para as categorias utilizadas.

## 13. Resultados principais

### Ranking

Top-5 conceitos com threshold `0.22`:

```text
1. an animal          133
2. a pet              109
3. a food product     101
4. a person           100
5. a human face        93
```

### Sensibilidade do threshold

```text
0.22 → 1.254 pares acima do corte
0.23 →   639
0.24 →   316
0.25 →   158
```

### Busca semântica

Foram executadas 8 consultas:

```text
a person
an animal
a food product
a person outdoors
a child playing sports
a person using an electronic device
a relaxing outdoor scene
a happy social moment
```

A qualidade variou conforme a consulta. Conceitos visuais mais diretos, como `an animal`, apresentaram recuperação muito consistente, enquanto composições relacionais mais específicas, como `a person using an electronic device`, apresentaram correspondências apenas parciais.

## 14. Conclusões

O experimento mostrou que o CLIP permite extrair informação semântica de um corpus visual sem treinar um novo classificador.

O mecanismo central foi:

```text
imagem ──→ embedding visual ┐
                            ├─ cosine similarity
texto  ──→ embedding textual┘
```

Os resultados também demonstraram que:

1. o score CLIP deve ser tratado como similaridade, não como probabilidade;
2. thresholds precisam ser calibrados para o corpus;
3. consultas simples tendem a ser mais consistentes;
4. consultas compostas podem ser satisfeitas apenas parcialmente;
5. a busca semântica consegue trabalhar até com descrições abstratas;
6. inspeção visual continua sendo importante para avaliar a qualidade das recuperações;
7. deduplicação exata não elimina necessariamente near-duplicates.

## 15. Resumo para estudo

### CLIP em uma frase

CLIP aprende um espaço no qual **imagem e texto semanticamente compatíveis ficam próximos**.

### Fluxo

```text
IMAGEM
 ↓
Vision Transformer
 ↓
visual_projection
 ↓
embedding 512-D
 ↓
normalização L2
        \
         → cosine similarity

TEXTO
 ↓
tokenizer
 ↓
input_ids + attention_mask
 ↓
Text Transformer
 ↓
text_projection
 ↓
embedding 512-D
 ↓
normalização L2
```

### Conceitos importantes

```text
Embedding
→ representação vetorial

Cosine similarity
→ mede alinhamento angular entre vetores

Contrastive learning
→ aproxima pares corretos e afasta pares incorretos

Zero-shot
→ utilizar conceitos não treinados como classes específicas da tarefa

Semantic search
→ recuperar imagens por significado de uma consulta textual

Padding
→ completa sequências para permitir batching

Attention mask
→ impede que padding seja tratado como conteúdo real
```

## 16. Checklist da rubrica — Atividade 2

- [x] ADS-16 com subconjunto representativo de pelo menos 500 imagens
- [x] Corpus final com 600 imagens
- [x] Deduplicação exata por SHA-256
- [x] CLIP pré-treinado sem fine-tuning
- [x] Embeddings visuais e textuais
- [x] Normalização L2
- [x] Cosine similarity
- [x] Pelo menos 20 descrições — foram utilizadas 24
- [x] Ranking por frequência semântica
- [x] Score médio de similaridade
- [x] Threshold definido e justificado
- [x] Análise de sensibilidade do threshold
- [x] Visualização dos 5 conceitos mais frequentes com exemplos
- [x] Busca semântica Top-5
- [x] Pelo menos 8 consultas — foram utilizadas 8
- [x] Consultas genéricas, específicas, concretas e abstratas
- [x] Análise documentada das recuperações
- [x] Explicação do alinhamento imagem–texto
- [x] Explicação do pré-treinamento contrastivo
- [x] Explicação de zero-shot / semantic retrieval
- [x] Comparação CLIP × BERT
- [x] Explicação de padding
- [x] Explicação de attention mask
- [x] Limitações e análise crítica